In [2]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
GPUs available: []


In [3]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU enabled with memory growth")
else:
    print("No GPU detected")


No GPU detected


In [ ]:
X_midi = np.load("../models/X_midi.npy")
y_midi = np.load("../models/y_midi.npy")

print("Loaded X_midi shape:", X_midi.shape)
print("Loaded y_midi shape:", y_midi.shape)


In [ ]:
MAX_SEQUENCES = 200000   # Safe value even on laptop GPU

X_midi = X_midi[:MAX_SEQUENCES]
y_midi = y_midi[:MAX_SEQUENCES]

print("Using sequences:", X_midi.shape[0])


In [ ]:
X_midi = X_midi.reshape(
    X_midi.shape[0],
    X_midi.shape[1],
    1
)

print("Reshaped X_midi:", X_midi.shape)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


In [ ]:
model = Sequential([
    LSTM(256, input_shape=(X_midi.shape[1], 1), return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dense(256, activation="relu"),
    Dense(len(np.unique(y_midi)), activation="softmax")
])


In [ ]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam"
)

model.summary()


In [ ]:
history = model.fit(
    X_midi,
    y_midi,
    epochs=20,
    batch_size=128,   # GPU allows larger batch
)


In [ ]:
model.save("../models/midi_lstm_model.h5")
print("✅ MIDI LSTM model saved successfully")
